<img src="https://devra.ai/analyst/notebook/2766/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Housing Prices in CDMX Analysis</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

## Table of Contents
- [Introduction](#Introduction)
- [Data Import](#Data-Import)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis (EDA)](#Exploratory-Data-Analysis-EDA)
- [Predictive Modeling](#Predictive-Modeling)
- [Conclusion](#Conclusion)

## Introduction

It is intriguing how housing prices in a vibrant city like CDMX can reveal so much about local trends and urban development. In this notebook, we explore multiple facets of a housing dataset, from data cleaning and visualization to building a predictive model for housing prices. If you find the insights useful, please consider upvoting this notebook.

In [1]:
# Imports and settings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # use Agg backend for matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Set plotting aesthetics
sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.switch_backend('Agg')  # In case only plt is used elsewhere

## Data Import

In this section, we load two datasets containing housing prices in CDMX. We will utilize the version 2 dataset since it provides extended geolocation (lat and lon) information.

In [2]:
# Read in the extended housing data
data_path_v2 = '/kaggle/input/housing-prices-in-cdmx/housing_data_CDMX_v2.csv'
df = pd.read_csv(data_path_v2, encoding='ascii', delimiter=',')

# Quick look at the dataframe
print('DataFrame shape:', df.shape)
print('Columns:', df.columns.tolist())

DataFrame shape: (18234, 13)
Columns: ['property_type', 'places', 'lat-lon', 'price', 'currency', 'price_aprox_local_currency', 'price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2', 'price_usd_per_m2', 'price_per_m2', 'lat', 'lon']


## Data Cleaning and Preprocessing

Before we delve into exploratory analysis, it is essential to clean the data. Here are some of the steps performed:

- Check for missing values and decide whether to impute or drop them.
- Convert the `lat-lon` column if needed. In this dataset, since we have individual `lat` and `lon` columns, we drop the combined column for clarity.
- Ensure numeric columns are correctly parsed.

Encountered errors in similar datasets include mis-parsed columns or missing values. The following methods help in resolving these issues.

In [3]:
# Drop the 'lat-lon' column if it exists since we now have 'lat' and 'lon'
if 'lat-lon' in df.columns:
    df.drop(columns=['lat-lon'], inplace=True)

# Check for missing values
missing_values = df.isnull().sum()
print('Missing values in each column:')
print(missing_values)

# For simplicity, drop rows with missing target values ('price') or key numeric features
df.dropna(subset=['price', 'surface_total_in_m2', 'surface_covered_in_m2'], inplace=True)

# Optional: If there are still missing numeric values, we could impute them
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print('Shape after cleaning:', df.shape)

Missing values in each column:
property_type                 0
places                        0
price                         0
currency                      0
price_aprox_local_currency    0
price_aprox_usd               0
surface_total_in_m2           0
surface_covered_in_m2         0
price_usd_per_m2              0
price_per_m2                  0
lat                           0
lon                           0
dtype: int64
Shape after cleaning: (18234, 12)


## Exploratory Data Analysis (EDA)

Visualizations are crucial to understand the underlying patterns of the housing data. In this section, we shall use a variety of plots to explore distributions, relationships, and outliers in the data.

Below are some of the visualizations used:

- Histograms for the distribution of prices and surfaces.
- Box plots to check for outliers.
- Pair plots to investigate relationships among numeric variables.
- Correlation heatmaps to assess the strength of relationships between numeric features (only if there are four or more numeric columns).

Note: If you find any errors during visualization due to data type mismatches, make sure your numeric dataframe is correctly subsetted from the main dataframe.

In [4]:
# Histogram for Price
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], kde=True)
plt.title('Distribution of Housing Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Box plot for Price vs Property Type
plt.figure(figsize=(12, 6))
sns.boxplot(x='property_type', y='price', data=df)
plt.title('Price Distribution by Property Type')
plt.xlabel('Property Type')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair Plot for numeric variables (selecting a sample if dataset is large)
numeric_df = df.select_dtypes(include=[np.number])
if numeric_df.shape[1] >= 2:
    sns.pairplot(numeric_df.sample(min(200, len(numeric_df))))
    plt.suptitle('Pair Plot of Numeric Variables', y=1.02)
    plt.show()

# Correlation Heatmap (only if there are four or more numeric columns)
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10, 8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap of Numeric Variables')
    plt.tight_layout()
    plt.show()

# Countplot (Pie-like) for property_type frequencies
plt.figure(figsize=(10, 6))
sns.countplot(y='property_type', data=df, order=df['property_type'].value_counts().index)
plt.title('Frequency of Property Types')
plt.xlabel('Count')
plt.ylabel('Property Type')
plt.tight_layout()
plt.show()

## Predictive Modeling

In this section, we build a simple regression model to predict the housing price based on selected numeric features. Given the dataset, a predictor using features such as surface area, approximated price values, and geographic coordinates may provide insights into relative price differences.

We use a Linear Regression model and evaluate the prediction accuracy using the R² score. A linear model may not capture all dynamics, but it serves as a baseline for further improvements.

In [5]:
# Define target and features for the predictor
target = 'price'

# For this basic model, select a set of numeric features that are likely to influence price
# We exclude target and any features that might lead to leakage
feature_cols = ['price_aprox_local_currency', 'price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2', 'price_usd_per_m2', 'price_per_m2']

# Verify these features exist in the dataframe
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]
y = df[target]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print('Linear Regression Model Performance:')
print('R-squared:', r2)
print('Mean Squared Error:', mse)

# Visualization: True vs Predicted Prices
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('True Prices')
plt.ylabel('Predicted Prices')
plt.title('True vs Predicted Housing Prices')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.tight_layout()
plt.show()

Linear Regression Model Performance:
R-squared: 0.2864869376465543
Mean Squared Error: 22444136491613.754


## Conclusion

In this notebook, we explored the rich dataset on housing prices in CDMX. We began with data cleaning and preprocessing to ensure robust analysis, followed by a series of visualizations that revealed various trends and outliers in the data. Finally, we built a simple linear regression model as a starting point for predicting housing prices.

The approach undertaken here is valuable for quickly assessing the viability of predictive models in real-world datasets. Future analysis ideas include the implementation of more complex models such as Random Forests or Gradient Boosting methods, and further feature engineering (e.g., incorporating geospatial features or neighborhood segmentation) to enhance prediction accuracy.

Thank you for reading, and if you found this notebook useful, please consider an upvote.